# Parte II — Programação Linear · o código dos capítulos 07 a 15

**Este caderno não contém o algoritmo.** Ele busca o código publicado do handbook e chama as
mesmas funções que o `pytest` do repositório verifica. Se o código mudar, o caderno muda junto:
não existe segunda cópia para envelhecer — é a regra da
[ADR 0016](https://github.com/GHDaru/operationalresearchaibook/blob/main/adr/0016-cadernos-colab-sem-deriva.md).

Rode as células em ordem. No Colab, `Ctrl+F9` roda tudo.

> **Ele roda igual na sua máquina.** O Colab é conveniência, não dependência: o experimento é o
> script em `po-zero/`, que roda em qualquer CPU sem licença paga.


In [ ]:
# 1. Traz o código publicado. Sem magias do IPython (`!git`, `%cd`): Python puro
#    roda igual no Colab e no seu terminal — e é o que o teste consegue executar.
import subprocess, sys
from pathlib import Path

URL = "https://github.com/GHDaru/operationalresearchaibook"
RAIZ = Path("operationalresearchaibook")

if not RAIZ.exists():
    subprocess.run(["git", "clone", "--depth", "1", URL, str(RAIZ)], check=True)

ETAPA = RAIZ / "po-zero" / "etapa-05-parte2"
sys.path.insert(0, str(ETAPA.resolve()))
print("código em:", ETAPA.resolve())


## Capítulo 12 — a montadora, e os preços que já estavam no quadro


In [ ]:
# 2. A instância-fio atravessa os capítulos 07 a 15. Ela é LIDA do arquivo,
#    nunca redigitada — um número muda num lugar só.
from dual import (montadora, precos_sombra, dual_explicito,
                  faixa_de_validade, faixa_de_custo, confere_faixa)
from fractions import Fraction as F

lucros, restricoes, ficha = montadora()
primal = precos_sombra(lucros, restricoes)
print("plano ótimo:", primal["ponto"], " lucro:", primal["valor"])
print("preços-sombra:", primal["precos"])


In [ ]:
# 3. O dual, montado e resolvido como problema PRÓPRIO — não lido do quadro.
#    Ler o preço no quadro do primal é circular: usa o mesmo cálculo para
#    produzir e para conferir. Chegar ao mesmo valor por dois caminhos é que
#    é verificação.
d = dual_explicito(lucros, restricoes)
print("y =", d["y"], " valor do dual:", d["valor_dual"], f"({d['pivos']} pivôs)")
assert F(primal["valor"]) == F(d["valor_dual"]), "dualidade forte NÃO confere"
print("dualidade forte confere ✔")


## O telefonema — erre antes de ver o certo

Sexta-feira, 16h. Um fornecedor oferece **10 CPUs a R$ 45** cada. O quadro diz que a CPU vale
**R$ 50**.

**Antes de rodar a próxima célula, decida:** você compra? E quanto ganha?

Escreva a sua resposta em algum lugar. A célula seguinte roda o **seu** raciocínio e depois o
que de fato acontece — nesta ordem, de propósito.


In [ ]:
# 4. Primeiro o raciocínio do reflexo. Depois a realidade.
from quadro import Restricao, resolver

PRECO, LOTE = 45, 10

reflexo = LOTE * (int(primal['precos']['CPUs']) - PRECO)
print(f"o reflexo diz: {LOTE} × (50 − {PRECO}) = R$ {reflexo} de ganho")

alt = [Restricao(list(r.coefs), r.sinal, r.b + (LOTE if i == 0 else 0), r.rotulo)
       for i, r in enumerate(restricoes)]
novo = F(resolver(lucros, alt)["valor"])
ganho = novo - F(primal["valor"])
resultado = ganho - LOTE * PRECO
print(f"o que acontece: lucro vai de {primal['valor']} para {novo} — ganho real R$ {ganho}")
print(f"custo: {LOTE} × R$ {PRECO} = R$ {LOTE * PRECO}")
print(f"resultado: R$ {resultado}")
assert resultado == -350, f"esperado prejuízo de 350, veio {resultado}"
print()
print("Nenhuma conta do reflexo estava errada. O que faltou foi a FAIXA.")


## Capítulo 13 — a faixa, que é a metade que quase ninguém publica


In [ ]:
# 5. Até onde o preço vale — por álgebra exata, conferida por um segundo caminho
#    que põe o estoque na fronteira e um pouco além.
esperado = {"CPUs": ("6", "12"), "pentes de memória de 16 GB": ("10", "20")}
for i in range(len(restricoes)):
    f = faixa_de_validade(lucros, restricoes, i)
    c = confere_faixa(lucros, restricoes, i, f)
    print(f"{f['componente']}: hoje {f['estoque_atual']} · faixa [{f['pode_cair_ate']}, {f['pode_subir_ate']}]")
    assert (f["pode_cair_ate"], f["pode_subir_ate"]) == esperado[f["componente"]]
    for lado in ("piso", "teto"):
        assert c[lado]["preco_acerta_na_fronteira"] and c[lado]["preco_erra_alem"]
print("faixas conferidas nos dois sentidos ✔")


## Mexa aqui

Troque `MEU_LOTE` abaixo e rode de novo. A faixa das CPUs vai de **6 a 12**, e o estoque hoje
é 10 — então só **2** unidades estão cobertas pela garantia.

Tente 1, 2, 3 e 8. O ponto em que o ganho para de crescer é a fronteira, e ela não é uma opinião.


In [ ]:
# 6. Sua vez. Mude o número e rode.
MEU_LOTE = 2      # <<< mexa aqui

alt = [Restricao(list(r.coefs), r.sinal, r.b + (MEU_LOTE if i == 0 else 0), r.rotulo)
       for i, r in enumerate(restricoes)]
z = F(resolver(lucros, alt)["valor"])
ganho = z - F(primal["valor"])
print(f"comprando {MEU_LOTE} CPU(s): lucro {z} · ganho R$ {ganho} · custo R$ {MEU_LOTE * PRECO} · resultado R$ {ganho - MEU_LOTE * PRECO}")
print(f"ganho por unidade comprada: R$ {ganho / MEU_LOTE if MEU_LOTE else 0}")


## O que o livro não mostra

A célula seguinte imprime **o código-fonte** da função que calcula a faixa. O capítulo explica a
ideia; aqui você lê a implementação, com o comentário que registra o defeito que ela já teve —
duas faixas erradas publicadas por medir a coisa errada.


In [ ]:
# 7. O algoritmo, lido em vez de descrito.
import inspect, dual
print(inspect.getsource(dual.faixa_de_validade))
